In [0]:
import pandas as pd

# 1. Definir las rutas de los archivos
url_ranking = "https://appscvsmovil.supercias.gob.ec/ranking/recursos/bi_ranking.csv"
df_ranking = pd.read_csv(url_ranking)

# display(df_ranking.head(100))
# print(df_ranking.columns)

# 2. Cargar solo las columnas necesarias de cada archivo
# Del ranking tomamos los indicadores y el expediente para unir los datos

# display(df_ranking[df_ranking['expediente'] == 1])

# cols_ranking = [
#      'expediente', 'posicion_general','anio', 'ciiu_n1', 'ciiu_n6', 'cod_segmento', 'n_empleados', 'ingresos_ventas', 'activos', 'patrimonio', 'utilidad_an_imp', 'impuesto_renta', 'ingresos_totales','utilidad_ejercicio', 'utilidad_neta', 'liquidez_corriente', 'prueba_acida', 'cobertura_interes', 'apalancamiento', 'apalancamiento_financiero', 'rot_cartera', 'rot_activo_fijo' , 'rot_ventas', 'per_med_cobranza', 'per_med_pago', 'impac_gasto_a_v', 'impac_carga_finan', 'rent_neta_activo', 'margen_bruto', 'margen_operacional', 'rent_neta_ventas', 'rent_ope_patrimonio', 'rent_ope_activo', 'roe', 'roa', 'fortaleza_patrimonial', 'gastos_financieros', 'gastos_admin_ventas', 'depreciaciones', 'amortizaciones', 'costos_ventas_prod', 'deuda_total', 'deuda_total_c_plazo', 'total_gastos'
#      ]
# df_ranking = df_ranking[cols_ranking]
# display(df_ranking.head(100))


# --------------------------------------------------------------------


# 1. Definir las rutas de los archivos
url_compania = "https://appscvsmovil.supercias.gob.ec/ranking/recursos/bi_compania.csv"
df_compania = pd.read_csv(url_compania)

# display(df_compania.head(100))
# print(df_compania.columns)

# 2. Cargar solo las columnas necesarias de cada archivo

# display(df_compania[df_compania['expediente'] == 1])

cols_cia = ['expediente', 'ruc', 'nombre', 'tipo', 'pro_codigo', 'provincia']
df_compania = df_compania[cols_cia]
# display(df_compania.head(100))


# --------------------------------------------------------------------


# 1. Definir las rutas de los archivos
url_segmento = "https://appscvsmovil.supercias.gob.ec/ranking/recursos/bi_segmento.csv"
df_segmento = pd.read_csv(url_segmento)

# display(df_segmento.head(100))
# print(df_segmento.columns)

# 2. Cargar solo las columnas necesarias de cada archivo

cols_seg = ['id_segmento', 'segmento']
df_segmento = df_segmento[cols_seg]
# display(df_segmento.head(100))


# --------------------------------------------------------------------


# 1. Definir las rutas de los archivos
url_sector = "https://appscvsmovil.supercias.gob.ec/ranking/recursos/bi_ciiu.csv"
df_sector = pd.read_csv(url_sector)

# display(df_sector.head(100))
# print(df_sector.columns)

# 2. Cargar solo las columnas necesarias de cada archivo

cols_sector = ['ciiu', 'descripcion']
df_sector = df_sector[cols_sector]
# display(df_sector.head(100))


# --------------------------------------------------------------------
# --------------------------------------------------------------------
# 3. Unir ambos archivos usando la columna 'expediente'

# Aplicar TRIM (.str.strip()) a las columnas de unión
df_ranking['ciiu_n1'] = df_ranking['ciiu_n1'].astype(str).str.strip()
df_sector['ciiu'] = df_sector['ciiu'].astype(str).str.strip()

df_resultado = (
    df_ranking
     .merge(df_compania, on='expediente', how='inner')
     .merge(df_segmento, left_on='cod_segmento', right_on='id_segmento', how='left')
     .merge(df_sector, left_on='ciiu_n1', right_on='ciiu', how='left')
     .merge(df_sector, left_on='ciiu_n6', right_on='ciiu', how='left')

)
# display(df_resultado[df_resultado['expediente'] == 1])


# 4. Convertir el DataFrame de Pandas (df_final) a un DataFrame de Spark
# Esto es necesario para interactuar con el Catálogo de Databricks
spark_df = spark.createDataFrame(df_resultado)

# 2. Guardar la tabla en la ruta
spark_df.write.mode("overwrite").saveAsTable("lakehouse_gtyt_dev.db_bronze.supercias_ranking")

# display(spark.table("lakehouse_gtyt_dev.db_bronze.supercias_ranking"))

In [0]:
%skip
import pandas as pd

# 1. Definir los datos del glosario
glosario_datos = [
    # Datos de Compañía
    ("expediente", "Número identificador único de la compañía otorgado por la Superintendencia de Compañías, Valores y Seguros (SCVS)."),
    ("ruc", "Registro Único de Contribuyentes; número identificador tributario de la compañía otorgado por el Servicio de Rentas Internas (SRI)."),
    ("nombre", "Nombre o razón social de la compañía."),
    ("tipo", "Clasificación jurídica o forma legal de la compañía."),
    ("pro_codigo", "Código numérico asignado a la provincia del domicilio legal."),
    ("provincia", "Nombre de la provincia donde está domiciliada la compañía."),
    
    # Datos de Segmento
    ("id_segmento", "Número identificador del segmento al que pertenece una compañía."),
    ("segmento", "Descripción del segmento al que pertenece una compañía."),

    # Datos de CIIU
    ("ciiu", "Código de Clasificación Industrial Internacional Uniforme de la actividad económica (todos los niveles)."),
    ("descripcion", "Descripción del código ciiu."),
    
    # Datos Financieros e Indicadores
    ("anio", "Año fiscal en que se presenta el estado financiero."),
    ("posicion_general", "Número de la posición de la compañía."),
    ("cia_imvalores", "Marca si la compañía pertenece al sector de mercado de valores: 1 (Sí); 0 (No)."),
    ("id_estado_financiero", "Número identificador del estado financiero."),
    ("ingresos_ventas", "Sumatoria de los ingresos por ventas de acuerdo a lo registrado en el estado financiero."),
    ("activos", "Sumatoria de los activos de acuerdo a lo registrado en el estado financiero."),
    ("patrimonio", "Sumatoria del patrimonio de acuerdo a lo registrado en el estado financiero."),
    ("utilidad_an_imp", "Utilidad antes de impuestos; cifra resultante tras deducir costos y gastos (excepto impuestos) del total de ingresos."),
    ("impuesto_renta", "Valor aplicado sobre las ganancias obtenidas en el año fiscal registrado."),
    ("n_empleados", "Número de empleados registrado en el estado financiero."),
    ("ingresos_totales", "Sumatoria de todos los ingresos recibidos en el año fiscal registrado."),
    ("utilidad_ejercicio", "Ganancia obtenida por ventas de productos o servicios tras descontar costos de producción."),
    ("utilidad_neta", "Ganancia final tras descontar costos de producción, distribución, logística, gastos operativos, impuestos y obligaciones."),
    ("cod_segmento", "Número identificador del tipo de segmento al que pertenece la compañía."),
    ("ciiu_n1", "Código de Clasificación Industrial Internacional Uniforme a nivel 1 de la actividad económica."),
    ("ciiu_n6", "Código de Clasificación Industrial Internacional Uniforme a nivel 6 de la actividad económica."),
    ("liquidez_corriente", "Indicador de liquidez corriente calculado con los valores del estado financiero."),
    ("prueba_acida", "Indicador de prueba ácida calculado con los valores del estado financiero."),
    ("end_activo", "Indicador de endeudamiento del activo calculado con los valores del estado financiero."),
    ("end_patrimonial", "Indicador de endeudamiento patrimonial calculado con los valores del estado financiero."),
    ("end_activo_fijo", "Indicador de endeudamiento del activo fijo calculado con los valores del estado financiero."),
    ("end_corto_plazo", "Indicador de endeudamiento a corto plazo calculado con los valores del estado financiero."),
    ("end_largo_plazo", "Indicador de endeudamiento a largo plazo calculado con los valores del estado financiero."),
    ("cobertura_interes", "Indicador de cobertura de interés calculado con los valores del estado financiero."),
    ("apalancamiento", "Indicador de apalancamiento calculado con los valores del estado financiero."),
    ("apalancamiento_financiero", "Indicador de apalancamiento financiero calculado con los valores del estado financiero."),
    ("end_patrimonial_ct", "Indicador de endeudamiento patrimonial corriente calculado con los valores del estado financiero."),
    ("end_patrimonial_nct", "Indicador de endeudamiento patrimonial no corriente calculado con los valores del estado financiero."),
    ("apalancamiento_c_l_plazo", "Indicador de apalancamiento a corto y largo plazo calculado con los valores del estado financiero."),
    ("rot_cartera", "Indicador de rotación de cartera calculado con los valores del estado financiero."),
    ("rot_activo_fijo", "Indicador de rotación de activo fijo calculado con los valores del estado financiero."),
    ("rot_ventas", "Indicador de rotación de ventas calculado con los valores del estado financiero."),
    ("per_med_cobranza", "Indicador de período medio de cobranza calculado con los valores del estado financiero."),
    ("per_med_pago", "Indicador de período medio de pago calculado con los valores del estado financiero."),
    ("impac_gasto_a_v", "Indicador de impacto de gastos de administración y ventas calculado con los valores del estado financiero."),
    ("impac_carga_finan", "Indicador de impacto de la carga financiera calculado con los valores del estado financiero."),
    ("margen_bruto", "Indicador de margen bruto calculado con los valores del estado financiero."),
    ("margen_operacional", "Indicador de margen operacional calculado con los valores del estado financiero."),
    ("rent_neta_ventas", "Indicador de rentabilidad neta de ventas calculado con los valores del estado financiero."),
    ("rent_ope_patrimonio", "Indicador de rentabilidad operacional del patrimonio calculado con los valores del estado financiero."),
    ("rent_ope_activo", "Indicador de rentabilidad operacional del activo calculado con los valores del estado financiero."),
    ("roe", "Indicador de rentabilidad financiera calculado con los valores del estado financiero."),
    ("roa", "Indicador de rendimiento de los activos calculado con los valores del estado financiero."),
    ("fortaleza_patrimonial", "Indicador de fortaleza patrimonial calculado con los valores del estado financiero."),
    ("gastos_financieros", "Sumatoria de los gastos financieros registrados en el estado financiero."),
    ("gastos_admin_ventas", "Sumatoria de los gastos administrativos y de ventas registrados en el estado financiero."),
    ("depreciaciones", "Sumatoria de los gastos por depreciaciones registrados en el estado financiero."),
    ("amortizaciones", "Sumatoria de los gastos por amortizaciones registrados en el estado financiero."),
    ("costos_ventas_prod", "Sumatoria de los costos de ventas y producción registrados en el estado financiero."),
    ("deuda_total", "Sumatoria de todas las obligaciones financieras registradas en el estado financiero."),
    ("deuda_total_c_plazo", "Sumatoria de las obligaciones financieras corrientes (corto plazo) registradas en el estado financiero."),
    ("total_gastos", "Sumatoria total de los gastos registrados en el estado financiero.")
]

# 2. Crear el DataFrame
df_glosario = spark.createDataFrame(glosario_datos, ["termino", "definicion"])

# 3. Guardar como tabla permanente en Databricks (Delta Lake)
df_glosario.write.mode("overwrite").saveAsTable("lakehouse_gtyt_dev.db_bronze.supercias_glosario")

# 4. Mostrar el resultado
display(spark.table("lakehouse_gtyt_dev.db_bronze.supercias_glosario"))